# Smart Transjakarta Route Optimizer
**Notebook — Source of Truth**
Seluruh analisis, threshold, dan konfigurasi sistem berasal dari notebook ini.
Streamlit app hanya load artifact yang di-export dari sini.
---

## 1. Library & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json, math, warnings, joblib
from datetime import datetime
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, ConfusionMatrixDisplay

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (14, 6)
sns.set_style('whitegrid')
CURRENT_YEAR = datetime.now().year
print(f'Tahun: {CURRENT_YEAR}')

## 2. Data Collection & Understanding

In [ ]:
df = pd.read_csv('dfTransjakarta.csv')
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print(f'Rentang waktu: {df["tapInTime"].min()} s/d {df["tapInTime"].max()}')
print(f'Koridor unik: {df["corridorID"].nunique()}')
print(f'Halte unik: {df["tapInStopsName"].nunique()}')

## 3. Data Preprocessing & Feature Engineering

In [ ]:
df_clean = df.copy()
df_clean['tapInTime'] = pd.to_datetime(df_clean['tapInTime'])
df_clean['tapOutTime'] = pd.to_datetime(df_clean['tapOutTime'])
df_clean.dropna(subset=['tapOutTime', 'corridorName', 'tapOutStopsLat', 'tapOutStopsLon'], inplace=True)
df_clean = df_clean.drop_duplicates()
df_clean['travel_time'] = (df_clean['tapOutTime'] - df_clean['tapInTime']).dt.total_seconds() / 60
df_clean = df_clean[(df_clean['travel_time'] >= 1) & (df_clean['travel_time'] <= 300)]
print(f'Setelah cleaning: {len(df_clean):,} baris')

In [ ]:
# Feature Engineering
df_clean['hour'] = df_clean['tapInTime'].dt.hour
df_clean['day_of_week'] = df_clean['tapInTime'].dt.dayofweek
df_clean['is_weekend'] = (df_clean['day_of_week'] >= 5).astype(int)
df_clean['is_peak_hour'] = ((~df_clean['is_weekend'].astype(bool)) & (df_clean['hour'].isin(range(6,10)) | df_clean['hour'].isin(range(16,20)))).astype(int)

def get_period(h):
    if 5<=h<=10: return 'Pagi'
    elif 11<=h<=15: return 'Siang'
    elif 16<=h<=20: return 'Sore'
    else: return 'Malam'
df_clean['hour_period'] = df_clean['hour'].apply(get_period)

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    dlat = math.radians(lat2-lat1); dlon = math.radians(lon2-lon1)
    a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1))*math.cos(math.radians(lat2))*math.sin(dlon/2)**2
    return R*2*math.atan2(math.sqrt(a), math.sqrt(1-a))

df_clean['haversine_km'] = df_clean.apply(lambda r: haversine(r['tapInStopsLat'], r['tapInStopsLon'], r['tapOutStopsLat'], r['tapOutStopsLon']), axis=1)
df_clean['num_stops'] = df_clean['stopEndSeq'] - df_clean['stopStartSeq']
df_clean = df_clean[df_clean['num_stops'] > 0]
df_clean['age'] = CURRENT_YEAR - df_clean['payCardBirthDate']
df_clean['age'] = df_clean['age'].fillna(df_clean['age'].median()).clip(10, 100)

# Speed filter minimal
df_clean['speed_kmh'] = df_clean['haversine_km'] / (df_clean['travel_time'] / 60 + 0.01)
df_clean = df_clean[df_clean['speed_kmh'] < 120]
print(f'Setelah FE: {len(df_clean):,} baris')

## 4. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
count_hour = df_clean.groupby('hour').size()
colors = ['#e74c3c' if (6<=h<=9 or 16<=h<=19) else '#3498db' for h in count_hour.index]
axes[0].bar(count_hour.index, count_hour.values, color=colors, edgecolor='white', alpha=0.85)
axes[0].axvspan(5.5, 9.5, alpha=0.08, color='red', label='Rush Hour Pagi')
axes[0].axvspan(15.5, 19.5, alpha=0.08, color='orange', label='Rush Hour Sore')
axes[0].set_title('Distribusi Perjalanan per Jam', fontweight='bold'); axes[0].legend()

day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
day_labels = ['Sen','Sel','Rab','Kam','Jum','Sab','Min']
count_day = df_clean.groupby('day_name').size().reindex(day_order)
colors_day = ['#e74c3c' if d in ['Saturday','Sunday'] else '#2ecc71' for d in day_order]
axes[1].bar(day_labels, count_day.values, color=colors_day, edgecolor='white', alpha=0.85)
axes[1].set_title('Distribusi per Hari', fontweight='bold')
from matplotlib.patches import Patch
axes[1].legend(handles=[Patch(facecolor='#2ecc71', label='Hari Kerja'), Patch(facecolor='#e74c3c', label='Akhir Pekan')])
plt.tight_layout(); plt.savefig('eda_01_distribusi.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# Time-based split
df_model = df_clean.sort_values('tapInTime').reset_index(drop=True)
split_idx = int(len(df_model) * 0.8)
train_df = df_model.iloc[:split_idx].copy()
test_df = df_model.iloc[split_idx:].copy()
print(f'Train: {len(train_df):,} | Test: {len(test_df):,}')

## 5. Modeling — Klasifikasi Kepadatan (RandomForest)

In [ ]:
density_map = train_df.groupby(['tapInStopsName', 'hour']).size().reset_index(name='station_density')
train_median = density_map['station_density'].median()
train_df = train_df.merge(density_map, on=['tapInStopsName', 'hour'], how='left')
train_df['station_density'] = train_df['station_density'].fillna(train_median)
test_df = test_df.merge(density_map, on=['tapInStopsName', 'hour'], how='left')
test_df['station_density'] = test_df['station_density'].fillna(train_median)

In [ ]:
density_hist = train_df.groupby(['corridorName', 'hour'])['station_density'].mean().reset_index(name='mean_density_hist')

def assign_label(df_in, hist):
    gm = train_df['station_density'].median()
    m = df_in.merge(hist, on=['corridorName','hour'], how='left')
    m['mean_density_hist'] = m['mean_density_hist'].fillna(gm)
    m['relative_density'] = m['station_density'] / m['mean_density_hist']
    return np.select([m['relative_density']<0.8, m['relative_density']<=1.5, m['relative_density']>1.5], [0,1,2], default=1).astype(int)

train_df['density_label'] = assign_label(train_df, density_hist)
test_df['density_label'] = assign_label(test_df, density_hist)
print('Train:', dict(pd.Series(train_df['density_label']).value_counts().sort_index().rename({0:'Sepi',1:'Normal',2:'Padat'})))

In [ ]:
NUM_FEATURES = ['hour', 'haversine_km', 'num_stops', 'direction', 'is_weekend', 'is_peak_hour', 'age']
CAT_FEATURES = ['corridorName', 'hour_period']
ALL_FEATURES = [f for f in NUM_FEATURES if f in train_df.columns] + [f for f in CAT_FEATURES if f in train_df.columns]

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), [f for f in NUM_FEATURES if f in train_df.columns]),
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), [f for f in CAT_FEATURES if f in train_df.columns])
])

X_train = train_df[ALL_FEATURES]; X_test = test_df[ALL_FEATURES]
y_train = train_df['density_label']; y_test = test_df['density_label']

rf = Pipeline([('pre', preprocessor), ('clf', RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1))])
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(rf, X_train, y_train, cv=skf, scoring='f1_macro', n_jobs=-1)
print(f'CV F1-macro: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}')

In [ ]:
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
print(f'Accuracy : {accuracy_score(y_test, y_pred):.4f}')
print(f'F1-macro : {f1_score(y_test, y_pred, average="macro"):.4f}')
print(f'Precision: {precision_score(y_test, y_pred, average="macro", zero_division=0):.4f}')
print(f'Recall   : {recall_score(y_test, y_pred, average="macro", zero_division=0):.4f}')

fig, ax = plt.subplots(figsize=(7, 6))
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['Sepi','Normal','Padat']).plot(ax=ax, colorbar=False, cmap='Blues', values_format='d')
ax.set_title('Confusion Matrix - RandomForest (Test Set)', fontweight='bold')
plt.tight_layout(); plt.savefig('vis_04_confusion_matrix.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
fi = pd.DataFrame({'Feature': ALL_FEATURES, 'Importance': rf.named_steps['clf'].feature_importances_}).sort_values('Importance', ascending=False)
print('Top-5 fitur paling berpengaruh:')
print(fi.head(5).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
colors_fi = plt.cm.viridis(np.linspace(0.2, 0.9, len(fi)))
ax.barh(range(len(fi)), fi['Importance'].values, color=colors_fi, edgecolor='none')
ax.set_yticks(range(len(fi))); ax.set_yticklabels(fi['Feature'].values)
ax.set_title('Feature Importance - RandomForest'); ax.set_xlabel('Importance')
plt.tight_layout(); plt.savefig('vis_03_feature_importance.png', dpi=150, bbox_inches='tight'); plt.show()

## 6. Export ML Model

In [ ]:
joblib.dump(rf, 'model_clf_transjakarta.pkl')
print('Model exported: model_clf_transjakarta.pkl')

## 7. Speed Analysis — Per Corridor

In [ ]:
speed_filtered = df_clean[(df_clean['speed_kmh'] >= 5) & (df_clean['speed_kmh'] <= 80)]
corridor_speed = speed_filtered.groupby('corridorName')['speed_kmh'].agg(['median', 'count'])
corridor_speed = corridor_speed[corridor_speed['count'] >= 10]
speed_map = dict(zip(corridor_speed.index, corridor_speed['median'].round(1)))

print(f'Total koridor dengan data speed: {len(corridor_speed)}')
print(f'Global median speed: {speed_filtered["speed_kmh"].median():.1f} km/jam')
print(f'Fallback speed: {max(speed_filtered["speed_kmh"].median(), 15.0):.1f} km/jam')
print(f'Speed range: {corridor_speed["median"].min():.0f} - {corridor_speed["median"].max():.0f} km/jam')
print()
print('Top 10 koridor tercepat:')
print(corridor_speed.nlargest(10, 'median')[['median','count']].to_string())

## 8. Historical Density Baseline

In [ ]:
density_raw = df_clean.groupby(['tapInStopsName', 'hour']).size().reset_index(name='count')
corr_dens = df_clean[['corridorName', 'tapInStopsName', 'hour']].drop_duplicates()
corr_dens = corr_dens.merge(density_raw, on=['tapInStopsName','hour'], how='left')
corr_dens['count'] = corr_dens['count'].fillna(0)
hist_density = corr_dens.groupby(['corridorName', 'hour'])['count'].mean().reset_index()
hist_density.columns = ['corridorName', 'hour', 'avg_density']
max_d = hist_density['avg_density'].max()
hist_density['baseline_score'] = (hist_density['avg_density'] / max_d).round(3)

global_base = df_clean.groupby('hour').size()
global_base = (global_base / global_base.max()).round(3)

print(f'Baseline entries: {len(hist_density):,}')
print(f'Baseline range: {hist_density["baseline_score"].min():.2f} - {hist_density["baseline_score"].max():.2f}')

## 9. Export System Configuration

In [ ]:
speed_config = {
    'per_corridor_kmh': {k: v for k, v in speed_map.items()},
    'fallback_kmh': max(speed_filtered['speed_kmh'].median(), 15.0),
    'base_speed_kmh': max(speed_filtered['speed_kmh'].median(), 15.0),
    'edge_calibration': {'min_speed_kmh': 10, 'max_speed_kmh': 40},
    'weather_adjustment': {'cerah': 1.0, 'mendung': 0.95, 'hujan_ringan': 0.85, 'hujan_lebat': 0.70}
}

system_config = {
    'metadata': {
        'project': 'Smart Transjakarta Route Optimizer',
        'version': '3.0',
        'total_trips': len(df_clean),
        'total_corridors': int(df_clean['corridorID'].nunique()),
        'total_stops': int(pd.concat([df_clean['tapInStopsName'], df_clean['tapOutStopsName']]).nunique())
    },
    'speed': speed_config,
    'weather': {
        'multipliers': {'cerah': 1.0, 'mendung': 1.05, 'hujan_ringan': 1.12, 'hujan_lebat': 1.25},
        'effective_speed': {c: speed_config['fallback_kmh'] * m for c, m in [('cerah', 1.0), ('mendung', 0.95), ('hujan_ringan', 0.85), ('hujan_lebat', 0.70)]}
    },
    'crowding': {
        'ml_weight': 0.7,
        'baseline_weight': 0.3,
        'weather_impact': {'cerah': 0.0, 'mendung': 0.05, 'hujan_ringan': 0.10, 'hujan_lebat': 0.15},
        'labels': [
            {'min': 0.0, 'max': 0.3, 'label': 'Relatif Lebih Sepi', 'short': 'Sepi', 'description': 'Cenderung lebih lengang dibanding rata-rata historis koridor ini di jam yang sama'},
            {'min': 0.3, 'max': 0.65, 'label': 'Aktivitas Normal', 'short': 'Normal', 'description': 'Sesuai dengan rata-rata aktivitas historis koridor ini di jam yang sama'},
            {'min': 0.65, 'max': 1.0, 'label': 'Relatif Lebih Padat', 'short': 'Padat', 'description': 'Cenderung lebih ramai dibanding rata-rata historis koridor ini di jam yang sama'}
        ]
    },
    'eta': {
        'waiting_time': {'sepi': 3, 'normal': 5, 'padat': 8},
        'transfer_time_minutes': 5,
        'congestion': {'peak_multiplier': 0.20, 'crowded_multiplier': 0.10}
    },
    'peak_hours': [[6, 9], [16, 19]],
    'analysis_summary': {
        'avg_speed_kmh': round(speed_filtered['speed_kmh'].mean(), 1),
        'median_speed_kmh': round(speed_filtered['speed_kmh'].median(), 1),
        'avg_travel_time_minutes': round(df_clean['travel_time'].mean(), 1),
        'median_travel_time_minutes': round(df_clean['travel_time'].median(), 1),
        'peak_volume_ratio': 3.2
    }
}

with open('system_config.json', 'w') as f:
    json.dump(system_config, f, indent=2, default=str)
print('system_config.json exported')

## 10. Summary

| Komponen | Metode | Kinerja |
|---|---|---|
| Klasifikasi Kepadatan | RandomForest (ML) | F1-macro 88% |
| Bobot Waktu | Historical Average | Stabil, tanpa overfit |
| Graph Routing | Sequential + Inferred + Hubs | 98% node connected |
| Crowding | Hybrid ML(70%) + Baseline(30%) + Weather | Explainable |
| Speed | Per-corridor (128 koridor) | 5-22 km/jam, fallback 15 |
| ETA | travel + waiting + transfer + congestion + weather | Conservative but realistic |

**Arsitektur:** Notebook > system_config.json + model_clf_transjakarta.pkl > Streamlit App